# Причинный вывод на практике: Lalonde NSW
## Часть 2. Наблюдательный датасет: наивная оценка и смещение

Заменяем рандомизированный control на CPS (общий опрос населения, не связан
с NSW) - имитация типичной продуктовой задачи без RCT.

Эталон из Части 1 (`benchmark.json`): ATE = 1,794 (95% CI [551, 3,038]).

## 2. Метод

Наивный diff-in-means, без поправок, точка отсчёта, показывающая масштаб
смещения без коррекции.

Ожидание: участники NSW - недавно безработные, моложе и беднее среднего
CPS-респондента. Наивная оценка, вероятно, занизит эффект или даст
отрицательный знак.

In [1]:
!pip install -q causaldata --break-system-packages 2>/dev/null || pip install -q causaldata

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 32.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import sys
sys.path.append('.')

import pandas as pd
import numpy as np
from causaldata import nsw_mixtape, cps_mixtape

from ci_utils import smd_table, diff_in_means, load_benchmark, append_result

nsw = nsw_mixtape.load_pandas().data
cps = cps_mixtape.load_pandas().data

print(f'NSW: {nsw.shape[0]} наблюдений (experimental)')
print(f'CPS: {cps.shape[0]} наблюдений (общий опрос населения, все treat=0)')

NSW: 445 наблюдений (experimental)
CPS: 15992 наблюдений (общий опрос населения, все treat=0)


## 3. Сборка датасета

Treated - 185 из NSW. Control - весь CPS (`treat=0` по построению), родной
NSW-control не используется.

In [3]:
treated = nsw[nsw['treat'] == 1].copy()
control = cps.copy()

df = pd.concat([treated, control], ignore_index=True)

print(f'Treated (NSW): {len(treated)}')
print(f'Control (CPS): {len(control)}')
print(f'Итоговый датасет: {df.shape[0]} наблюдений')
print()
print(df['treat'].value_counts())

Treated (NSW): 185
Control (CPS): 15992
Итоговый датасет: 16177 наблюдений

treat
0    15992
1      185
Name: count, dtype: int64


control 185 vs 15,992 - не сбой: control здесь весь доступный пул опроса, а
не специально набранная под эксперимент группа.

## 4. Баланс

Рандомизации нет — ожидаем дисбаланс, в отличие от Части 1.

In [4]:
covariates = ['age', 'educ', 'black', 'hisp', 'marr', 'nodegree', 're74', 're75']
balance = smd_table(df, 'treat', covariates)
balance.round(2)

,mean_treat,mean_control,smd
covariate,,,
age,25.82,33.23,-0.80
educ,10.35,12.03,-0.68
black,0.84,0.07,2.43
hisp,0.06,0.07,-0.05
marr,0.19,0.71,-1.23
nodegree,0.71,0.30,0.90
re74,2095.57,14016.80,-1.57
re75,1532.06,13650.80,-1.75


`re74`/`re75` (SMD −1.57/−1.75) - доход NSW-участников на порядок ниже CPS.
`age` (−0.80), `marr` (−1.23), `black` (+2.43) - тоже далеко за порогом.

Каждая ковариата - confounder: связана и с попаданием в treatment, и с
`re78`.

## 5. Наивная оценка на испорченном датасете

In [5]:
naive = diff_in_means(df, 're78', 'treat')

print(f"ATE (diff-in-means): {naive['ate']:,.0f}")
print(f"SE: {naive['se']:,.0f}")
print(f"95% CI: [{naive['ci_low']:,.0f}, {naive['ci_high']:,.0f}]")
print(f"p-value: {naive['p_value']:.2e}")

ATE (diff-in-means): -8,498
SE: 712
95% CI: [-9,893, -7,102]
p-value: 1.07e-32


## 6. Сравнение с эталоном из Части 1

In [6]:
benchmark = load_benchmark('benchmark.json')

bias = naive['ate'] - benchmark['ate']

print(f"Эталон (RCT, Часть 1):        {benchmark['ate']:,.0f}")
print(f"Наивная оценка (NSW+CPS):     {naive['ate']:,.0f}")
print(f"Смещение (bias):              {bias:,.0f}")
print(f"Смещение в SE эталона:        {bias / benchmark['se']:.1f} SE")

Эталон (RCT, Часть 1):        1,794
Наивная оценка (NSW+CPS):     -8,498
Смещение (bias):              -10,292
Смещение в SE эталона:        -16.3 SE


Наивная оценка ≈ -8,498 против эталона +1,794 - смещение разворачивает знак,
на порядок больше самого эффекта. Прямое следствие дисбаланса ковариат:
control в среднем богаче и старше ещё до начала программы. Классический
результат LaLonde (1986) - обоснование для matching/PSM/PSW/DML в
следующих частях.

## Что дальше

Результат сохранён в `results.csv`.

Дальше - Часть 3: matching / PSM / PSW на этом же датасете.

In [7]:
append_result(naive, label='Naive diff-in-means (NSW+CPS)', path='results.csv')

,method,ate,se,ci_low,ci_high,p_value,n_treat,n_control
0,Naive diff-in-means (NSW+CPS),-8497.515625,712.020724,-9893.155036,-7101.876214,1.074814e-32,185,15992
